In [ ]:
# ============================================================
# CELL 1 — KẾT NỐI DATASET VÀ CẤU HÌNH EDA
# ============================================================

from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import duckdb

# Dataset đã làm sạch trên Kaggle
DATASET_DIR = Path(
    "/kaggle/input/datasets/huy2008/taobao-clean"
)

# Tìm chính xác các file Parquet
PARQUET_FILES = sorted(
    DATASET_DIR.rglob("part_*.parquet")
)

if len(PARQUET_FILES) != 201:
    raise FileNotFoundError(
        f"Dự kiến 201 file Parquet, tìm thấy "
        f"{len(PARQUET_FILES)}. "
        "Kiểm tra dataset_sources trong kernel-metadata.json."
    )

# Thư mục kết quả
OUTPUT_DIR = Path(
    "/kaggle/working/taobao_eda_outputs"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEMP_DIR = Path(
    "/kaggle/working/duckdb_temp"
)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

# Cấu hình DuckDB
con = duckdb.connect(
    database="/kaggle/working/taobao_eda.duckdb"
)

con.execute("SET memory_limit = '4GB'")
con.execute("SET threads = 2")
con.execute(f"SET temp_directory = '{TEMP_DIR}'")

# Danh sách đường dẫn thực tế của 201 file
parquet_paths = [str(p) for p in PARQUET_FILES]

# Tạo VIEW từ Parquet

con.from_parquet(parquet_paths).create_view(
    "taobao",
    replace=True
)

# Kiểm tra VIEW đã được tạo
print("VIEW taobao created successfully!")
print("Number of Parquet files:", len(parquet_paths))

print("\nSchema:")
print(con.execute("DESCRIBE taobao").df())

# Cấu hình biểu đồ
plt.rcParams.update({
    "figure.figsize": (12, 7),
    "figure.dpi": 120,
    "savefig.dpi": 600,
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.titlesize": 17,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "axes.grid": True,
    "grid.alpha": 0.20,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

COLORS = {
    "pv": "#3977B8",
    "fav": "#9363B5",
    "cart": "#E0A03B",
    "buy": "#2B9A75"
}

def save_chart(fig, filename):
    fig.savefig(
        OUTPUT_DIR / f"{filename}.png",
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )
    fig.savefig(
        OUTPUT_DIR / f"{filename}.pdf",
        bbox_inches="tight",
        facecolor="white"
    )
    plt.show()
    plt.close(fig)

print("=" * 60)
print("CELL 1 — TAOBAO EDA CONFIGURATION")
print("=" * 60)
print("Dataset:", DATASET_DIR)
print("Parquet files:", len(PARQUET_FILES))
print("First file:", PARQUET_FILES[0].name)
print("Last file:", PARQUET_FILES[-1].name)
print("Output:", OUTPUT_DIR)
print("Chart quality: PNG 600 DPI + PDF Vector")
print("DuckDB: Connected")
print("\nCELL 1 COMPLETED SUCCESSFULLY")

In [ ]:
# CELL 2 — BEHAVIOR DISTRIBUTION
# Thống kê 4 hành vi trên toàn bộ dữ liệu đã làm sạch

behavior_order = ["pv", "fav", "cart", "buy"]

behavior_df = con.execute("""
    SELECT behavior, COUNT(*) AS interactions
    FROM taobao
    GROUP BY behavior
""").df()

behavior_counts = (
    behavior_df.set_index("behavior")
    .reindex(behavior_order, fill_value=0)["interactions"]
)

total_interactions = int(behavior_counts.sum())
expected_total = 100_095_231

assert total_interactions == expected_total, (
    f"Expected {expected_total:,}, got {total_interactions:,}"
)

behavior_stats = pd.DataFrame({
    "behavior": behavior_order,
    "interactions": behavior_counts.to_numpy(dtype="int64"),
    "percentage": behavior_counts.to_numpy() / total_interactions * 100
})

behavior_stats.to_csv(
    OUTPUT_DIR / "01_behavior_distribution.csv",
    index=False
)

fig, ax = plt.subplots(figsize=(11, 7))

bars = ax.bar(
    behavior_stats["behavior"],
    behavior_stats["interactions"],
    color=[COLORS[b] for b in behavior_order],
    width=0.62
)

for bar, count, pct in zip(
    bars,
    behavior_stats["interactions"],
    behavior_stats["percentage"]
):
    ax.annotate(
        f"{count:,}\n({pct:.2f}%)",
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 7),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=11
    )

ax.set_title("Taobao User Behavior Distribution")
ax.set_xlabel("Behavior Type")
ax.set_ylabel("Number of Interactions")
ax.set_ylim(0, behavior_stats["interactions"].max() * 1.17)
ax.ticklabel_format(axis="y", style="plain")
ax.grid(axis="x", visible=False)

fig.tight_layout()
save_chart(fig, "01_behavior_distribution")

print(behavior_stats.to_string(index=False))
print(f"\nTotal interactions: {total_interactions:,}")
print("CELL 2 COMPLETED SUCCESSFULLY")


In [ ]:
# ============================================================
# CELL 3 — DAILY INTERACTIONS
# Taobao EDA | Full cleaned dataset | UTC+8
# ============================================================

import matplotlib.dates as mdates

# 1. Thống kê số tương tác theo ngày (Asia/Shanghai, UTC+8)
# Timestamp trong dữ liệu là Unix timestamp (giây, UTC).
# Cộng 8 giờ trước khi lấy ngày để khớp với temporal split.

daily_df = con.execute("""
    SELECT
        CAST(
            DATE_TRUNC(
                'day',
                to_timestamp(timestamp) + INTERVAL '8 hours'
            ) AS DATE
        ) AS date,
        COUNT(*) AS interactions
    FROM taobao
    GROUP BY 1
    ORDER BY 1
""").df()

# 2. Chuẩn hóa ngày và kiểm tra dữ liệu
daily_df["date"] = pd.to_datetime(daily_df["date"])
daily_df["interactions"] = daily_df["interactions"].astype("int64")

expected_dates = pd.date_range(
    start="2017-11-25",
    end="2017-12-03",
    freq="D"
)

assert len(daily_df) == 9, (
    f"Expected 9 days, got {len(daily_df)}"
)

assert daily_df["date"].tolist() == expected_dates.tolist(), (
    "Unexpected date range!"
)

assert daily_df["interactions"].sum() == 100_095_231, (
    "Daily interaction total does not match cleaned dataset!"
)

# 3. Xuất bảng thống kê CSV
daily_df.to_csv(
    OUTPUT_DIR / "02_daily_interactions.csv",
    index=False
)

# 4. Vẽ biểu đồ
fig, ax = plt.subplots(figsize=(13, 7))

ax.plot(
    daily_df["date"],
    daily_df["interactions"],
    color="#3977B8",
    linewidth=2.5,
    marker="o",
    markersize=8,
    label="Daily Interactions"
)

ax.fill_between(
    daily_df["date"],
    daily_df["interactions"],
    color="#3977B8",
    alpha=0.12
)

# Hiển thị số lượng tương tác trên từng điểm
for date, count in zip(
    daily_df["date"],
    daily_df["interactions"]
):
    ax.annotate(
        f"{count / 1_000_000:.2f}M",
        xy=(date, count),
        xytext=(0, 10),
        textcoords="offset points",
        ha="center",
        fontsize=10
    )

ax.set_title(
    "Taobao Daily User Interactions (Nov 25 - Dec 03, 2017)"
)
ax.set_xlabel("Date (UTC+8)")
ax.set_ylabel("Number of Interactions")

ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))

ax.set_ylim(
    0,
    daily_df["interactions"].max() * 1.18
)

ax.ticklabel_format(
    axis="y",
    style="plain"
)

ax.grid(axis="x", visible=False)
ax.legend(loc="upper left")

fig.autofmt_xdate(rotation=0)
fig.tight_layout()

# 5. Lưu PNG 600 DPI và PDF vector
save_chart(fig, "02_daily_interactions")

# 6. In kết quả
print("\nDAILY INTERACTIONS (UTC+8)")
print("-" * 45)
print(daily_df.to_string(index=False))

print("\nTotal interactions:",
      f"{daily_df['interactions'].sum():,}")

print("Number of days:", len(daily_df))
print("CELL 3 COMPLETED SUCCESSFULLY")


In [ ]:
# ============================================================
# CELL 4 — USER DEGREE DISTRIBUTION (LOG-LOG)
# Full Taobao Cleaned Dataset | Bipartite User-Item Graph
# ============================================================

import time

start_time = time.perf_counter()

# 1. Tính số lượng item KHÁC NHAU cho mỗi user
# COUNT(DISTINCT item_id) tránh đếm lặp tương tác.
# DuckDB xử lý trực tiếp từ Parquet, không nạp toàn bộ
# 100 triệu dòng vào Pandas.

print("Computing user degrees on full cleaned dataset...")

con.execute("""
    CREATE OR REPLACE TABLE user_degrees AS
    SELECT
        user_id,
        COUNT(DISTINCT item_id) AS degree
    FROM taobao
    GROUP BY user_id
""")

# 2. Thống kê phân bố degree
# Mỗi degree k tương ứng với số user có degree = k.

user_degree_dist = con.execute("""
    SELECT
        degree,
        COUNT(*) AS user_count
    FROM user_degrees
    GROUP BY degree
    ORDER BY degree
""").df()

# 3. Các thống kê tổng quan
user_degree_stats = con.execute("""
    SELECT
        COUNT(*) AS total_users,
        MIN(degree) AS min_degree,
        MAX(degree) AS max_degree,
        AVG(degree) AS mean_degree,
        MEDIAN(degree) AS median_degree,
        SUM(degree) AS unique_user_item_edges
    FROM user_degrees
""").df()

# 4. Kiểm tra dữ liệu
assert not user_degree_dist.empty
assert (user_degree_dist["degree"] > 0).all()

total_users = int(user_degree_stats["total_users"].iloc[0])
unique_edges = int(
    user_degree_stats["unique_user_item_edges"].iloc[0]
)

assert int(user_degree_dist["user_count"].sum()) == total_users

print(f"Total users: {total_users:,}")
print(f"Unique user-item edges: {unique_edges:,}")

# 5. Xuất CSV
user_degree_dist.to_csv(
    OUTPUT_DIR / "03_user_degree_distribution.csv",
    index=False
)

user_degree_stats.to_csv(
    OUTPUT_DIR / "03_user_degree_statistics.csv",
    index=False
)

# 6. Vẽ biểu đồ Log-Log
fig, ax = plt.subplots(figsize=(12, 7))

ax.scatter(
    user_degree_dist["degree"],
    user_degree_dist["user_count"],
    s=22,
    alpha=0.70,
    color="#3977B8",
    edgecolors="none",
    label="Observed User Degree Distribution"
)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_title(
    "Taobao User Degree Distribution (Log-Log)"
)
ax.set_xlabel("User Degree (Unique Items)")
ax.set_ylabel("Number of Users")

ax.grid(
    True,
    which="both",
    alpha=0.20,
    linestyle="--"
)

ax.legend(loc="upper right")

fig.tight_layout()

# 7. Lưu PNG 600 DPI và PDF vector
save_chart(fig, "03_user_degree_distribution")

# 8. In thống kê
print("\nUSER DEGREE STATISTICS")
print("-" * 50)
print(user_degree_stats.to_string(index=False))

elapsed = time.perf_counter() - start_time

print(f"\nExecution time: {elapsed:.2f} seconds")
print("CELL 4 COMPLETED SUCCESSFULLY")

In [ ]:
# ============================================================
# CELL 5 — ITEM DEGREE DISTRIBUTION (LOG-LOG)
# Full Taobao Cleaned Dataset | Bipartite User-Item Graph
# ============================================================

import time

start_time = time.perf_counter()

# 1. Tính số lượng user KHÁC NHAU cho mỗi item
# COUNT(DISTINCT user_id) tránh đếm trùng các tương tác.
# DuckDB xử lý trực tiếp trên toàn bộ dữ liệu Parquet.

print("Computing item degrees on full cleaned dataset...")

con.execute("""
    CREATE OR REPLACE TABLE item_degrees AS
    SELECT
        item_id,
        COUNT(DISTINCT user_id) AS degree
    FROM taobao
    GROUP BY item_id
""")

# 2. Tính phân bố degree
# Mỗi degree k tương ứng số item có đúng k user khác nhau.

item_degree_dist = con.execute("""
    SELECT
        degree,
        COUNT(*) AS item_count
    FROM item_degrees
    GROUP BY degree
    ORDER BY degree
""").df()

# 3. Thống kê tổng quan
item_degree_stats = con.execute("""
    SELECT
        COUNT(*) AS total_items,
        MIN(degree) AS min_degree,
        MAX(degree) AS max_degree,
        AVG(degree) AS mean_degree,
        MEDIAN(degree) AS median_degree,
        SUM(degree) AS unique_user_item_edges
    FROM item_degrees
""").df()

# 4. Kiểm tra tính chính xác
assert not item_degree_dist.empty
assert (item_degree_dist["degree"] > 0).all()

total_items = int(item_degree_stats["total_items"].iloc[0])
unique_edges = int(
    item_degree_stats["unique_user_item_edges"].iloc[0]
)

assert int(item_degree_dist["item_count"].sum()) == total_items, (
    "Item count mismatch!"
)

# Tổng degree của User phải bằng tổng degree của Item
# vì cả hai đều đếm cùng tập cạnh User-Item duy nhất.

user_edges = int(
    con.execute("""
        SELECT SUM(degree)
        FROM user_degrees
    """).fetchone()[0]
)

assert unique_edges == user_edges, (
    f"Edge mismatch: item={unique_edges:,}, "
    f"user={user_edges:,}"
)

print(f"Total items: {total_items:,}")
print(f"Unique user-item edges: {unique_edges:,}")
print("User-Item edge consistency: PASSED")

# 5. Xuất CSV
item_degree_dist.to_csv(
    OUTPUT_DIR / "04_item_degree_distribution.csv",
    index=False
)

item_degree_stats.to_csv(
    OUTPUT_DIR / "04_item_degree_statistics.csv",
    index=False
)

# 6. Vẽ biểu đồ Log-Log
fig, ax = plt.subplots(figsize=(12, 7))

ax.scatter(
    item_degree_dist["degree"],
    item_degree_dist["item_count"],
    s=22,
    alpha=0.70,
    color="#2B9A75",
    edgecolors="none",
    label="Observed Item Degree Distribution"
)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_title(
    "Taobao Item Degree Distribution (Log-Log)"
)
ax.set_xlabel("Item Degree (Unique Users)")
ax.set_ylabel("Number of Items")

ax.grid(
    True,
    which="both",
    alpha=0.20,
    linestyle="--"
)

ax.legend(loc="upper right")

fig.tight_layout()

# 7. Lưu PNG 600 DPI và PDF vector
save_chart(fig, "04_item_degree_distribution")

# 8. In kết quả
print("\nITEM DEGREE STATISTICS")
print("-" * 50)
print(item_degree_stats.to_string(index=False))

elapsed = time.perf_counter() - start_time

print(f"\nExecution time: {elapsed:.2f} seconds")
print("CELL 5 COMPLETED SUCCESSFULLY")

In [ ]:
# ============================================================
# CELL 6 — DISTINCT BEHAVIORS PER USER
# Full Taobao Cleaned Dataset | 4 Behavior Types
# ============================================================

import time

start_time = time.perf_counter()

behavior_order = ["pv", "fav", "cart", "buy"]

print("Computing distinct behaviors per user...")

# 1. Tính số loại hành vi khác nhau của mỗi user
# COUNT(DISTINCT behavior) đếm loại hành vi,
# không đếm số lần tương tác.

con.execute("""
    CREATE OR REPLACE TABLE user_behavior_diversity AS
    SELECT
        user_id,
        COUNT(DISTINCT behavior) AS distinct_behaviors
    FROM taobao
    GROUP BY user_id
""")

# 2. Thống kê số user có 1, 2, 3 hoặc 4 loại hành vi

behavior_diversity_df = con.execute("""
    SELECT
        distinct_behaviors,
        COUNT(*) AS user_count
    FROM user_behavior_diversity
    GROUP BY distinct_behaviors
    ORDER BY distinct_behaviors
""").df()

# Đảm bảo hiển thị đủ 4 mức kể cả khi một mức có 0 user
behavior_diversity_df = (
    behavior_diversity_df
    .set_index("distinct_behaviors")
    .reindex([1, 2, 3, 4], fill_value=0)
    .reset_index()
)

# 3. Tính tỷ lệ phần trăm

total_users = int(behavior_diversity_df["user_count"].sum())

behavior_diversity_df["percentage"] = (
    behavior_diversity_df["user_count"]
    / total_users * 100
)

# 4. Kiểm tra dữ liệu

assert total_users > 0

assert behavior_diversity_df["user_count"].ge(0).all()

assert abs(
    behavior_diversity_df["percentage"].sum() - 100
) < 1e-8

# Đối chiếu với số lượng user ở Cell 4
user_count_cell4 = int(
    con.execute("SELECT COUNT(*) FROM user_degrees").fetchone()[0]
)

assert total_users == user_count_cell4, (
    f"User count mismatch: "
    f"Cell 6 = {total_users:,}, "
    f"Cell 4 = {user_count_cell4:,}"
)

print("User count consistency: PASSED")

# 5. Xuất thống kê CSV

behavior_diversity_df.to_csv(
    OUTPUT_DIR / "05_distinct_behaviors_per_user.csv",
    index=False
)

# 6. Vẽ biểu đồ

fig, ax = plt.subplots(figsize=(11, 7))

bars = ax.bar(
    behavior_diversity_df["distinct_behaviors"],
    behavior_diversity_df["user_count"],
    color=["#3977B8", "#9363B5", "#E0A03B", "#2B9A75"],
    width=0.60
)

# Hiển thị số lượng user và tỷ lệ trên mỗi cột
for bar, count, pct in zip(
    bars,
    behavior_diversity_df["user_count"],
    behavior_diversity_df["percentage"]
):
    ax.annotate(
        f"{count:,}\n({pct:.2f}%)",
        xy=(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height()
        ),
        xytext=(0, 7),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=11
    )

ax.set_title(
    "Number of Distinct Behaviors per User"
)

ax.set_xlabel("Number of Distinct Behavior Types")
ax.set_ylabel("Number of Users")

ax.set_xticks([1, 2, 3, 4])

ax.set_ylim(
    0,
    behavior_diversity_df["user_count"].max() * 1.18
)

ax.ticklabel_format(axis="y", style="plain")
ax.grid(axis="x", visible=False)

fig.tight_layout()

# 7. Lưu PNG 600 DPI và PDF vector
save_chart(fig, "05_distinct_behaviors_per_user")

# 8. In thống kê

print("\nDISTINCT BEHAVIORS PER USER")
print("-" * 55)

print(
    behavior_diversity_df.to_string(index=False)
)

print(f"\nTotal users: {total_users:,}")

average_behaviors = (
    behavior_diversity_df["distinct_behaviors"]
    * behavior_diversity_df["user_count"]
).sum() / total_users

print(
    f"Average distinct behaviors per user: "
    f"{average_behaviors:.4f}"
)

elapsed = time.perf_counter() - start_time

print(f"Execution time: {elapsed:.2f} seconds")
print("CELL 6 COMPLETED SUCCESSFULLY")

In [ ]:
# ============================================================
# CELL 7 — ORIGINAL VS CLEANED DATASET
# Taobao EDA | Data Cleaning Comparison
# ============================================================

import time

start_time = time.perf_counter()

# 1. Số liệu từ bước tiền xử lý toàn bộ dataset
original_count = 100_150_807
expected_cleaned_count = 100_095_231

# 2. Kiểm tra số lượng thực tế từ dataset đã làm sạch
print("Validating cleaned dataset size...")

cleaned_count = int(
    con.execute("""
        SELECT COUNT(*)
        FROM taobao
    """).fetchone()[0]
)

assert cleaned_count == expected_cleaned_count, (
    f"Cleaned count mismatch: "
    f"expected={expected_cleaned_count:,}, "
    f"actual={cleaned_count:,}"
)

# 3. Tính các chỉ số
removed_count = original_count - cleaned_count

retention_rate = (
    cleaned_count / original_count * 100
)

removal_rate = (
    removed_count / original_count * 100
)

assert removed_count >= 0

assert original_count == cleaned_count + removed_count

assert abs(
    retention_rate + removal_rate - 100
) < 1e-8

# 4. Tạo bảng thống kê
comparison_df = pd.DataFrame({
    "dataset": ["Original", "Cleaned"],
    "interactions": [
        original_count,
        cleaned_count
    ]
})

cleaning_stats = pd.DataFrame({
    "metric": [
        "Original Interactions",
        "Cleaned Interactions",
        "Removed Interactions",
        "Retention Rate (%)",
        "Removal Rate (%)"
    ],
    "value": [
        original_count,
        cleaned_count,
        removed_count,
        retention_rate,
        removal_rate
    ]
})

# 5. Xuất CSV
comparison_df.to_csv(
    OUTPUT_DIR / "06_original_vs_cleaned.csv",
    index=False
)

cleaning_stats.to_csv(
    OUTPUT_DIR / "06_cleaning_statistics.csv",
    index=False
)

# 6. Vẽ biểu đồ
fig, ax = plt.subplots(figsize=(11, 7))

colors = ["#3977B8", "#2B9A75"]

bars = ax.bar(
    comparison_df["dataset"],
    comparison_df["interactions"],
    color=colors,
    width=0.50
)

# Hiển thị số tương tác trên từng cột
for bar, count in zip(
    bars,
    comparison_df["interactions"]
):
    ax.annotate(
        f"{count:,}",
        xy=(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height()
        ),
        xytext=(0, 9),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold"
    )

# Thông tin tiền xử lý
summary_text = (
    f"Removed: {removed_count:,} interactions\n"
    f"Retention: {retention_rate:.4f}%\n"
    f"Removal: {removal_rate:.4f}%"
)

ax.text(
    0.5,
    0.76,
    summary_text,
    transform=ax.transAxes,
    ha="center",
    va="center",
    fontsize=12,
    bbox=dict(
        boxstyle="round,pad=0.6",
        facecolor="white",
        edgecolor="#CCCCCC",
        alpha=0.95
    )
)

ax.set_title(
    "Taobao Dataset: Original vs Cleaned"
)

ax.set_xlabel("Dataset Version")
ax.set_ylabel("Number of Interactions")

ax.set_ylim(
    0,
    original_count * 1.18
)

ax.ticklabel_format(axis="y", style="plain")
ax.grid(axis="x", visible=False)

fig.tight_layout()

# 7. Lưu PNG 600 DPI và PDF vector
save_chart(fig, "06_original_vs_cleaned")

# 8. In thống kê
print("\nORIGINAL VS CLEANED DATASET")
print("-" * 55)

print(comparison_df.to_string(index=False))

print(f"\nOriginal interactions: {original_count:,}")
print(f"Cleaned interactions:  {cleaned_count:,}")
print(f"Removed interactions:  {removed_count:,}")
print(f"Retention rate:        {retention_rate:.4f}%")
print(f"Removal rate:          {removal_rate:.4f}%")

elapsed = time.perf_counter() - start_time

print(f"\nExecution time: {elapsed:.2f} seconds")
print("CELL 7 COMPLETED SUCCESSFULLY")